# Ollama에서 Hugging Face GGUF 모델 사용하기

Hugging Face에 공개된 GGUF 모델은 파일을 별도 폴더에 내려받고 `Modelfile`로 등록하지 않아도 `hf.co/{사용자}/{저장소}:{양자화}` 형식으로 Ollama에 가져올 수 있다. 이 노트북에서는 모델을 Ollama cache에 한 번 준비한 뒤 Python API와 LangChain에서 같은 model ID를 재사용한다.


## GGUF 포맷

[GGUF](https://huggingface.co/docs/hub/en/gguf)는 모델 가중치뿐 아니라 tokenizer와 실행에 필요한 metadata를 하나의 파일에 담는 포맷이다. llama.cpp와 Ollama 같은 경량 추론 엔진이 빠르게 읽을 수 있으며, 여러 양자화 방식을 지원한다.

- **단일 파일**: 가중치와 metadata를 함께 보관해 배포하기 쉽다.
- **양자화 지원**: 4비트·5비트처럼 낮은 정밀도로 저장해 파일 크기와 추론 메모리를 줄일 수 있다.
- **실행 엔진 호환**: llama.cpp 계열 도구와 Ollama에서 사용할 수 있다.

`Q5_K_M`은 5비트 계열 양자화 방식이다. 일반적으로 더 낮은 bit 수는 메모리를 줄이지만 원본 가중치와의 차이가 커질 수 있다.

## 선수 조건과 패키지 준비

RunPod에서 `01_ollama.ipynb`를 먼저 완료해 Ollama server가 실행 중이어야 한다. Python의 `ollama` package는 새 server를 만드는 도구가 아니라 같은 Pod의 `http://localhost:11434` server에 요청하는 client이다.

In [1]:
%pip install -U ollama langchain-ollama

import ollama
from langchain_ollama import ChatOllama


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 7.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 744.6/744.6 kB 10.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 60.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [langchain-ollama][langchain-core]col]
Note: you may need to restart the kernel to use updated packages.


## Hugging Face GGUF 모델 바로 사용하기

`heegyu/EEVE-Korean-Instruct-10.8B-v1.0-GGUF:latest` 저장소는 현재 Ollama의 Hugging Face manifest 처리에서 호환 오류가 발생한다. 따라서 같은 `yanolja/EEVE-Korean-Instruct-10.8B-v1.0` 기반의 `Q5_K_M` GGUF인 [SourPineapple 저장소](https://huggingface.co/SourPineapple/EEVE-Korean-Instruct-10.8B-v1.0-Q5_K_M-GGUF)를 사용한다.

`ollama pull`은 GGUF를 임의의 `/workspace` 폴더에 저장하는 명령이 아니다. Ollama가 관리하는 model cache에 최초 한 번 내려받아 이후 `generate()`와 `chat()`이 같은 model ID를 사용할 수 있게 한다. [Hugging Face의 Ollama 가이드](https://huggingface.co/docs/hub/en/ollama)는 저장소 뒤에 `:Q5_K_M`처럼 tag를 붙여 원하는 양자화를 선택하는 형식을 제공한다.

In [2]:
MODEL_ID = (
    'hf.co/SourPineapple/'
    'EEVE-Korean-Instruct-10.8B-v1.0-Q5_K_M-GGUF:Q5_K_M'
)


### Ollama cache에 모델 준비하기

Python client의 `pull()`에 model ID를 전달하면 Ollama server가 Hugging Face에서 GGUF를 내려받아 자체 cache로 관리한다. 최초 실행은 파일 크기만큼 시간이 필요하지만 이후 실행에서는 저장된 layer를 재사용한다.

In [3]:
import ollama

pull_result = ollama.pull(MODEL_ID)
print(pull_result)

status='success' completed=None total=None digest=None


## `ollama.generate()`로 한 번 생성하기

`generate()`는 하나의 prompt를 전달해 assistant 역할 구분 없이 텍스트를 생성한다. `model`과 `prompt`를 전달하며, 반환값의 `response`에 생성된 본문이 들어 있다.

In [5]:
response = ollama.generate(
    model = MODEL_ID,
    prompt = "Meta사의 Ollama가 뭐야?"
)
print(response["response"])

<Olma>는 터키어로 '존재하다' 또는 '있다'를 의미하는 동사입니다. Meta의 Olma는 Meta와 Olma라는 두 단어 조합입니다. Meta는 다양한 플랫폼과 서비스를 운영하는 미국의 다국적 기술 회사로, 페이스북, 인스타그램, 왓츠앱 등이 있습니다.

Meta의 Olma라는 표현은 Meta가 Meta의 플랫폼이나 서비스에서 무언가를 존재시키거나 존재하게 만드는 것을 의미할 수 있습니다. 이 용어는 Meta의 다양한 기술 및 플랫폼이 어떻게 사용되는지와 관련되어 Meta의 전략, 정책 또는 제품 개발에 관한 토론이나 기사에서 사용될 수 있습니다.

예를 들어:

"최근 Meta는 소셜 미디어 플랫폼에서 긍정적인 사용자 경험을 촉진하기 위해 온라인 괴롭힘 Olma를 감소시키는 데 집중하고 있습니다."

이 문장에서 'Olma'는 Meta의 플랫폼에서 괴롭힘의 존재를 감소시키려는 노력을 나타냅니다. Meta의 Olma는 Meta가 사용자 경험을 개선하고 플랫폼의 긍정적인 환경을 조성하기 위해 취하는 조치들을 강조합니다.


## `ollama.chat()`으로 역할이 있는 대화하기

`chat()`은 `system`, `user`, `assistant` 역할이 있는 message 목록을 전달한다. 반환값의 `message.content`에서 assistant 답변을 꺼낸다.

In [7]:
chat_response = ollama.chat(
    model = MODEL_ID,
    messages = [
        {
            "role": "system",
            "content:" : "너는 현업 AI 엔지니어로 일하고 있고, 조언을 해주는 역할이야. 관련 질문에 대해서 간략하게 2문단 이내로 대답해.."
        },
        {
            "role": "user",
            "content": "신입 AI 엔지니어가 되려면 무엇을 준비해야해?"
        }
    ]
)
print(chat_response["message"]["content"])

신입 AI 엔지니어가 되기 위해서는 컴퓨터 과학, 수학, 인공지능 분야의 탄탄한 기초를 갖추고 있어야 합니다. 다음은 준비하는 데 도움이 될 단계들입니다:

1. 고등학교 및 대학교 학업:

- 고등학교 때 컴퓨터 과학, 수학, 과학 과목에 집중하세요. 컴퓨터 프로그래밍, 데이터 구조, 프로그래밍 언어, 선형대수, 미적분, 물리학 등을 공부하세요.

- 대학교에서는 컴퓨터 과학, 컴퓨터 공학, 전기공학, 또는 관련 분야의 학사 학위를 취득하세요. 이러한 학과들은 소프트웨어 개발, 데이터 구조, 알고리즘, 운영 체제, 프로그래밍 언어를 가르칩니다.

2. 인공지능(AI) 및 머신러닝(ML) 공부:

- 대학에서 인공지능, 머신러닝, 또는 데이터 과학과 관련된 과정이나 클럽에 참여하세요.

- 딥러닝, 강화학습, 자연어 처리, 컴퓨터 비전과 같은 AI 및 ML 주제에 대해 스스로 학습하세요. 온라인 코스를 듣거나 AI 및 ML 관련 책을 읽어 지식을 확장하세요.

3. 프로그래밍 언어 능력 개발:

- Python은 AI 및 ML 개발에 가장 널리 사용되는 프로그래밍 언어입니다. 기본부터 시작하여 고급 주제와 라이브러리(예: NumPy, SciPy, TensorFlow, PyTorch, Keras 등)를 배워보세요.

- AI 개발에 사용되는 인기 프로그래밍 언어인 C++, Java, R, Julia도 익숙해지세요.

4. 프로젝트 경험 쌓기:

- 작은 AI 프로젝트부터 시작하여 점차 복잡성을 높여 가세요. 이를 통해 기술을 연습하고 포트폴리오를 구축할 수 있습니다.

- 실제 AI 문제를 해결하고 기술을 선보이는 챌린지, 해커톤, 경연 대회에 참여하세요.

- 오픈소스 AI 프로젝트에 기여하고, 오픈소스 AI 도구 및 프레임워크를 만들어 기술적 전문성을 더욱 향상시키세요.

5. 데이터 과학 및 분석 기술 습득:

- AI 개발자는 데이터를 수집, 정리, 분석하여 유용한 통찰을 도출하는 방법을 알아야 합니다.

- 데이터 시각화, 데이터 마이닝, 데이터 웨어하

## LangChain `ChatOllama`로 같은 모델 호출하기

`ChatOllama`는 같은 Ollama server와 model ID를 LangChain의 Chat Model 인터페이스로 감싼다. GGUF를 다시 내려받거나 다른 모델로 변환하는 과정과 관계가 없다.

In [11]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model = MODEL_ID,
    temperature = 0.2,

)
langchain_response = llm.invoke("대한민국 24절기에 대해서 간단히 설명해줘.")

print(langchain_response.content)

대한민국 24절기는 전통적으로 한국의 달력에 따라 24개의 중요한 날들로 구성된 것으로, 계절의 변화와 농사의 중요한 단계를 기념합니다. 이들은 음력 달력에 기반하고 있으며, 태양력의 24절기와는 다릅니다.

대한민국 24절기는 다음과 같습니다:

1. 입춘 (立春): 음력 1월 21일경에 봄의 시작을 알리는 날입니다.
2. 우수 (雨水): 음력 2월 19일경에 겨울 추위가 끝나고 봄비가 내리는 것을 알리는 날입니다.
3. 경칩 (驚蟄): 음력 2월 20일경에 겨울잠을 자던 벌레들이 깨어나는 것을 알리는 날입니다.
4. 청명 (淸明): 음력 3월 21일경에 날씨가 맑아지고 농사일을 시작하는 것을 알리는 날입니다.
5. 곡우 (穀雨): 음력 4월 20일경에 봄비로 농작물이 자라는 것을 알리는 날입니다.
6. 입하 (立夏): 음력 5월 5일경에 여름의 시작을 알리는 날입니다.
7. 소만 (小滿): 음력 5월 21일경에 여름이 시작되고 낮이 길어지는 것을 알리는 날입니다.
8. 망종 (芒種): 음력 6월 5일경에 보리를 수확하고 모내기를 시작하는 것을 알리는 날입니다.
9. 하지 (夏至): 음력 6월 21일경에 낮이 가장 긴 날로, 여름의 정점을 알리는 날입니다.
10. 소서 (小暑): 음력 7월 7일경에 여름 더위가 시작되는 것을 알리는 날입니다.
11. 대서 (大暑): 음력 7월 23일경에 여름 더위가 가장 심한 것을 알리는 날입니다.
12. 입추 (立秋): 음력 8월 7일경에 가을의 시작을 알리는 날입니다.
13. 처서 (處暑): 음력 8월 23일경에 더위가 물러가고 시원한 바람이 부는 것을 알리는 날입니다.
14. 백로 (白露): 음력 9월 8일경에 이슬이 맺히고 가을이 시작됨을 알리는 날입니다.
15. 추분 (秋分): 음력 9월 23일경에 밤과 낮의 길이가 같아지는 것을 알리는 날입니다.
16. 한로 (寒露): 음력 10월 8일경에 서리가 내리고 추위가 시작되는 것을 알리는 날입니다.
17. 상강 (霜降): 음력 10월 23일경에 서리가 내리고 추위가 더 

## 선택 확장: 직접 `Modelfile`을 만드는 경우

두 번째 경로인 `GGUF 다운로드 → Modelfile 작성 → ollama create`는 model의 prompt template와 생성 parameter를 직접 바꿀 때만 사용한다. 기본 실습처럼 공개 GGUF를 그대로 실행할 때는 필요하지 않으므로 이 노트북의 필수 실행 단계에서 제외한다. 변환과 수동 등록 과정은 `hf_to_gguf.ipynb`에서 별도로 다룬다.

## 정리

- 기본 실습은 Hugging Face model ID를 Ollama cache에 준비하고 `ollama.generate()`로 호출한다.
- `/workspace`에 GGUF를 따로 내려받는 과정은 기본 실습에서 제외한다.
- `generate()`, `chat()`, `ChatOllama`는 같은 Ollama server와 같은 model을 서로 다른 입력 형식으로 호출한다.
- `Modelfile`은 공개 GGUF의 template나 parameter를 직접 바꿔야 할 때만 선택한다.